In [0]:
# =============================================================================
# Notebook: 02_generate_load_bronze
# Purpose : Simulates a raw data ingestion process. Generates synthetic retail
#           transaction data and writes it to the Bronze layer in Delta format.
#           Includes intentional data quality issues (nulls, duplicates) to 
#           test downstream validation and CDC logic.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
import random
from datetime import datetime, timedelta

# Set seed for reproducibility so the synthetic dataset remains consistent 
# across multiple pipeline runs, making debugging and ETL testing reliable.
random.seed(42)

# ---- Step 1: Configure synthetic data parameters ----
n = 5000
products = ['Shampoo', 'Conditioner', 'Face Cream', 'Lipstick', 'Serum', 'Sunscreen']
regions = ['North', 'South', 'East', 'West']
start_date = datetime(2024, 1, 1)

data = []

# ---- Step 2: Generate base transactional data ----
for i in range(1, n + 1):
    # Randomize dates to simulate transactions occurring over time
    order_date = start_date + timedelta(days=random.randint(0, 300))
    last_modified = start_date + timedelta(days=random.randint(0, 300), hours=random.randint(0, 23))
    
    # Intentionally inject ~1% NULL values into the 'amount' column.
    # This simulates real-world messy data to test our Silver layer validation checks.
    amount = round(random.uniform(5, 500), 2) if random.random() > 0.01 else None
    
    data.append((
        i,                                 # order_id
        random.randint(1000, 2000),        # customer_id (simulates repeat customers)
        random.choice(products),           # product
        random.choice(regions),            # region
        random.randint(1, 5),              # quantity
        amount,                            # amount (with intentional nulls)
        order_date,                        # order_date
        last_modified                      # last_modified timestamp
    ))

# ---- Step 3: Inject intentional duplicates ----
# Randomly duplicate 20 rows to simulate source system retries or exact-match 
# duplicates, testing our Bronze-to-Silver deduplication logic.
data += random.sample(data, 20)

# ---- Step 4: Define explicit schema ----
# Enforcing a strict schema prevents data type inference issues during DataFrame creation.
schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("region", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_date", TimestampType(), True),
    StructField("last_modified", TimestampType(), True),
])

df_orders = spark.createDataFrame(data, schema)

# ---- Step 5: Write to Bronze Layer (ADLS Gen2) ----
storage_account = "stretailcdcproj"
bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/retail_orders/"

# Save as Delta Lake format to support ACID transactions and time travel.
df_orders.write.mode("overwrite").format("delta").save(bronze_path)

print(f"✅ Loaded {df_orders.count()} rows into Bronze layer")
df_orders.show(5)


✅ Loaded 5020 rows into bronze layer
+--------+-----------+-----------+------+--------+------+-------------------+-------------------+
|order_id|customer_id|    product|region|quantity|amount|         order_date|      last_modified|
+--------+-----------+-----------+------+--------+------+-------------------+-------------------+
|       1|       1754|    Shampoo| North|       5|115.49|2024-02-27 00:00:00|2024-01-13 23:00:00|
|       2|       1616|    Shampoo| South|       5|120.17|2024-08-04 00:00:00|2024-01-17 00:00:00|
|       3|       1006|Conditioner|  West|       3|405.67|2024-08-02 00:00:00|2024-04-22 14:00:00|
|       4|       1094|   Lipstick| North|       3|171.61|2024-05-22 00:00:00|2024-03-20 06:00:00|
|       5|       1996|   Lipstick| North|       5|270.43|2024-06-25 00:00:00|2024-05-15 01:00:00|
+--------+-----------+-----------+------+--------+------+-------------------+-------------------+
only showing top 5 rows
